# 01 - Exploração dos dados brutos
Objetivo: entender a estrutura dos 3 CSVs do PNCP 2025 e identificar as sujeiras antes da limpeza.


In [7]:
import pandas as pd

compras = pd.read_csv("../data/raw/compras.csv", dtype=str)
itens = pd.read_csv("../data/raw/itens.csv", dtype=str)
resultados = pd.read_csv("../data/raw/resultados.csv", dtype=str)

print(compras.shape, itens.shape, resultados.shape)


(2042, 25) (7077, 14) (15896, 13)


---
## Tabela Compras


In [12]:
compras.info()

<class 'pandas.DataFrame'>
RangeIndex: 2042 entries, 0 to 2041
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   numero_controle_pncp        2042 non-null   str  
 1   ano_compra                  2042 non-null   str  
 2   sequencial_compra           2042 non-null   str  
 3   numero_compra               2042 non-null   str  
 4   processo                    2014 non-null   str  
 5   objeto_compra               2042 non-null   str  
 6   srp                         2042 non-null   str  
 7   cnpj_orgao                  2042 non-null   str  
 8   razao_social_orgao          2042 non-null   str  
 9   poder_id                    2042 non-null   str  
 10  esfera_id                   2042 non-null   str  
 11  uf_sigla                    2042 non-null   str  
 12  uf_nome                     2042 non-null   str  
 13  municipio                   2042 non-null   str  
 14  cod_ibge           

In [13]:
compras.head()


,numero_controle_pncp,ano_compra,sequencial_compra,numero_compra,processo,objeto_compra,srp,cnpj_orgao,razao_social_orgao,poder_id,...,nome_unidade,modalidade_id,modalidade_nome,modo_disputa,situacao_compra,valor_total_estimado,valor_total_homologado,data_publicacao,data_abertura_proposta,data_encerramento_proposta
0,00889834000108-1-000053/2023,2023,53,90037,0,Contratação de serviços continuados de comunic...,False,00889834000108,FUND COORD DE APERFEICOAMENTO DE PESSOAL DE NI...,E,...,MEC-FUCAPES-FUND.COORD.DE AP.NIV.SUPERIOR/DF,5,Concorrência - Presencial,Fechado,Divulgada no PNCP,20307804.45,"19.290.383,46",2025-01-06T11:51:06,06.03.2025,30.04.2025
1,32580400000100-1-000059/2024,2024,59,PE 009/2024,060/2024,[Portal de Compras Públicas] - AQUISIÇÃO DE EQ...,False,32580400000100,conselho regional dos tecnicos industriais de ...,E,...,Conselho Regional dos Técnicos Industriais de ...,6,Pregão - Eletrônico,Aberto,Divulgada no PNCP,589433.22,"420,562.14",2025-01-02T08:37:22,20250102,14/01/2025 09:00
2,06272868000127-1-000057/2024,2024,57,90025,128/2021,O objeto da presente licitação é a escolha da ...,False,06272868000127,CONSELHO REGIONAL DE ENFERMAGEM COREN MA,E,...,CONSELHO REGIONAL DE ENFERMAGEM - MA,6,Pregão - Eletrônico,Aberto-Fechado,Divulgada no PNCP,"250,782.10",151826.0,02/01/2025 09:03,02/01/2025,2025-01-14T09:00:00
3,00119784000171-1-000036/2024,2024,36,90011,0110029.0172/2024,Contratação de empresa especializada na presta...,False,119784000171,CONSELHO FEDERAL DE MEDICINA VETERINARIA,E,...,CONSELHO FEDERAL DE MEDICINA VETERINARIA,6,Pregão - Eletrônico,Aberto,Divulgada no PNCP,4612985.04,3718205.52,2025-01-02T09:03:51,2025-09-22T08:00:00,2025-10-07T10:00:00
4,00119784000171-1-000037/2024,2024,37,90012,-,Aquisição de Solução de Rede Local Sem Fio (WL...,False,00119784000171,CONSELHO FEDERAL DE MEDICINA VETERINARIA,E,...,CONSELHO FEDERAL DE MEDICINA VETERINARIA,6,Pregão - Eletrônico,Aberto,Divulgada no PNCP,"R$ 419.848,81",258905.0,02.01.2025,02/01/2025,20250114


### Colunas de `compras` (uma linha por contratação)

| Coluna | Significado | Observação |
|---|---|---|
| numero_controle_pncp | Identificador único da compra no PNCP; liga os 3 arquivos | Formato CNPJ-1-SEQUENCIAL/ANO; o prefixo é o CNPJ do órgão já limpo |
| ano_compra | Ano do número da compra | Pode ser anterior a 2025; a data que vale é `data_publicacao` |
| sequencial_compra | Número sequencial da compra no órgão | Igual à 3ª parte do `numero_controle_pncp` |
| numero_compra | Número da compra no sistema do órgão | Formatos livres (ex.: `90001`, `PE 009/2024`); não usado na análise |
| processo | Número do processo administrativo | 28 vazios; não usado na análise |
| objeto_compra | Descrição do que está sendo contratado | Texto livre; ajuda a identificar obras (pergunta 6) |
| srp | Indica Sistema de Registro de Preços (True/False) | Em SRP, um item tem vários fornecedores classificados |
| cnpj_orgao | CNPJ da unidade compradora | 539 distintos no bruto e 370 depois de limpo: máscaras e zeros perdidos |
| razao_social_orgao | Nome do órgão comprador | Mesmo órgão escrito de formas diferentes (maiúsculas, siglas) |
| poder_id | Poder do órgão: E, L, J ou N | Converter para o nome por extenso na `dim_orgao` |
| esfera_id | Esfera do órgão | Sempre `F` (Federal); não diferencia nada |
| uf_sigla | Sigla do estado da unidade | Base para criar a região (pergunta 5) |
| uf_nome | Nome do estado | |
| municipio | Cidade da unidade compradora | |
| cod_ibge | Código IBGE do município | |
| nome_unidade | Unidade específica dentro do órgão | Ex.: `IFES - CAMPUS SERRA` |
| modalidade_id | Código da modalidade de compra | 7 códigos; usar como chave (regra 9) |
| modalidade_nome | Nome da modalidade | Padronizar a partir do id |
| modo_disputa | Forma de disputa dos lances | 26 vazios; valores como `Aberto`, `Fechado`, `Não se aplica` |
| situacao_compra | Status do edital no portal | |
| valor_total_estimado | Valor total estimado da compra | Não usar: o grão da análise é o item |
| valor_total_homologado | Valor total homologado da compra | Não usar: o grão da análise é o item; formatos misturados |
| data_publicacao | Data de publicação do edital | Início do ciclo (pergunta 4); vários formatos de data |
| data_abertura_proposta | Início do prazo de propostas | 617 vazios; formatos misturados (`19.03.2025`, `2025-12-29T08:00:00`) |
| data_encerramento_proposta | Fim do prazo de propostas | 617 vazios; formatos misturados (`20250701`) |



---
## Tabela Itens

In [14]:
itens.info()

<class 'pandas.DataFrame'>
RangeIndex: 7077 entries, 0 to 7076
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   numero_controle_pncp     7077 non-null   str  
 1   numero_item              7077 non-null   str  
 2   descricao                7077 non-null   str  
 3   material_ou_servico      7077 non-null   str  
 4   categoria_item           7077 non-null   str  
 5   quantidade               7077 non-null   str  
 6   unidade_medida           7068 non-null   str  
 7   valor_unitario_estimado  7077 non-null   str  
 8   valor_total_estimado     7077 non-null   str  
 9   sigiloso                 7077 non-null   str  
 10  situacao_item            6985 non-null   str  
 11  tem_resultado            7077 non-null   str  
 12  catalogo_codigo_item     9 non-null      str  
 13  criterio_julgamento      7077 non-null   str  
dtypes: str(14)
memory usage: 774.2 KB


In [15]:
itens.head()

,numero_controle_pncp,numero_item,descricao,material_ou_servico,categoria_item,quantidade,unidade_medida,valor_unitario_estimado,valor_total_estimado,sigiloso,situacao_item,tem_resultado,catalogo_codigo_item,criterio_julgamento
0,00889834000108-1-000053/2023,1,Resenha de Notícia / Agenciamento de Informaçã...,Serviço,Não se aplica,3.0,UNIDADE,6769268.15,"20.307.804,45",False,Homologado,True,NaN,Melhor técnica
1,00119784000171-1-000036/2024,1,Assistência Médica - Hospitalar / Domiciliar c...,Serviço,Não se aplica,2724.0,UNIDADE,"R$ 1.693,46",4612985.04,False,Homologado,True,NaN,Menor preço
2,00119784000171-1-000037/2024,1,Ponto De Acesso,Material,Não se aplica,40.0,Unidade,8897.79,"355.911,60",False,Homologado,True,NaN,Menor preço
3,00119784000171-1-000037/2024,2,Equipamento Wireless,Material,Não se aplica,1.0,Unidade,25995.0,"25.995,00",False,Homologado,True,NaN,Menor preço
4,00119784000171-1-000037/2024,3,"Serviços de Instalação, Transição e Configuraç...",Serviço,Não se aplica,1.0,UND SERVIÇO TÉCNICO,"37,942.21",37942.21,False,Homologado,True,NaN,Menor preço


### Colunas de `itens` (uma linha por item licitado)

| Coluna | Significado | Observação |
|---|---|---|
| numero_controle_pncp | Compra à qual o item pertence | Liga com `compras` |
| numero_item | Número do item dentro da compra | Junto com `numero_controle_pncp`, identifica o item (grão da fato) |
| descricao | Descrição do item | Espaços extras e invisíveis (ex.: `  Tramadol  cloridrato`); usada no perfil do gasto |
| material_ou_servico | Classificação: Material ou Serviço | Base do perfil do gasto (pergunta 6) |
| categoria_item | Categoria do item | Sempre `Não se aplica`; não ajuda |
| quantidade | Quantidade licitada | Formatos misturados (`70.0`, `600,00`) |
| unidade_medida | Unidade da quantidade | 9 vazios; grafias variadas (`UNIDADE`, `Unidade`, `UNIDADE 0,00`) |
| valor_unitario_estimado | Preço unitário que o governo esperava pagar | Formatos misturados (`3.6`, `R$ 3,54`) |
| valor_total_estimado | Valor total esperado do item | Formatos misturados (`25497.5`, `20,400.00`); será recalculado (regra 5) |
| sigiloso | Orçamento sigiloso (True/False) | True = sem estimado utilizável; item rejeitado (regra 4) |
| situacao_item | Situação do item | 92 vazios |
| tem_resultado | Indica se o item tem fornecedor vencedor | Itens sem resultado ficam fora da fato (regra 3) |
| catalogo_codigo_item | Código do catálogo oficial | Só 9 preenchidos; não usado |
| criterio_julgamento | Critério para escolher o vencedor | Ex.: `Menor preço`, `Melhor técnica` |


---
## Tabela Resultados

In [16]:
resultados.info()

<class 'pandas.DataFrame'>
RangeIndex: 15896 entries, 0 to 15895
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   numero_controle_pncp       15896 non-null  str  
 1   numero_item                15896 non-null  str  
 2   sequencial_resultado       15896 non-null  str  
 3   ordem_classificacao_srp    3920 non-null   str  
 4   ni_fornecedor              15896 non-null  str  
 5   tipo_pessoa                15896 non-null  str  
 6   nome_fornecedor            15896 non-null  str  
 7   porte                      15700 non-null  str  
 8   quantidade_homologada      15896 non-null  str  
 9   valor_unitario_homologado  15896 non-null  str  
 10  valor_total_homologado     15896 non-null  str  
 11  data_resultado             15896 non-null  str  
 12  data_cancelamento          62 non-null     str  
dtypes: str(13)
memory usage: 1.6 MB


In [17]:
resultados.head()

,numero_controle_pncp,numero_item,sequencial_resultado,ordem_classificacao_srp,ni_fornecedor,tipo_pessoa,nome_fornecedor,porte,quantidade_homologada,valor_unitario_homologado,valor_total_homologado,data_resultado,data_cancelamento
0,00889834000108-1-000053/2023,1,1,1.0,51.593.855/0001-04,PJ,CRIATIVA DIGITAL COMUNICACOES LTDA,Demais,"R$ 3,00","6.430.127,82",19290383.46,10/03/2026 00:00,NaN
1,00119784000171-1-000036/2024,1,1,NaN,04487255000181,PJ,UNIMED SEGUROS SAUDE S/A,Demais,2724.0,1364.98,3718205.52,2025-10-31,2025-10-31T00:00:00
2,00119784000171-1-000036/2024,1,2,NaN,04.487.255/0001-81,PJ,unimed seguros saude s/a,Demais,2724.0,1364.98,"R$ 3.718.205,52",2025-10-31,NaN
3,00119784000171-1-000037/2024,1,1,NaN,04699854000169,PJ,GRG TECH ASSESSORIA EM INFORMATICA LTDA,Demais,40.0,5719.9,228796.0,20250226,NaN
4,00119784000171-1-000037/2024,2,1,NaN,04699854000169,PJ,GRG TECH ASSESSORIA EM INFORMATICA LTDA,Demais,1.0,900.0,900.0,2025-02-26,NaN


In [18]:
resultados["porte"].value_counts(dropna=False)


porte
ME                7101
Não Informado     4415
Demais            3893
NaN                196
SEM INFORMAÇÃO      60
0                   57
NÃO INFORMADO       53
-                   52
Null                51
EPP                  9
Não se aplica        9
Name: count, dtype: int64

### Colunas de `resultados` (uma linha por fornecedor classificado em cada item)

| Coluna | Significado | Observação |
|---|---|---|
| numero_controle_pncp | Compra à qual o resultado pertence | Liga com `compras` e `itens` |
| numero_item | Item ao qual o resultado pertence | Um item pode ter vários resultados (SRP) |
| sequencial_resultado | Número de sequência do resultado | Desempate na escolha do vencedor; comparar como número (2 antes de 10) |
| ordem_classificacao_srp | Posição do fornecedor no ranking (1 = vencedor) | Só 3.920 preenchidos; vazio vale 1 |
| ni_fornecedor | CNPJ ou CPF do fornecedor | Com e sem máscara (ex.: `620.053.323-72`); limpar conforme o tipo de pessoa |
| tipo_pessoa | PJ (empresa), PF (pessoa física) ou PE (estrangeiro) | Define se o documento rem 14 ou 11 caracteres ou letras |
| nome_fornecedor | Nome do fornecedor | Padronizar texto |
| porte | Porte da empresa (ME, EPP, Demais...) | 3 portes reais (ME, EPP, Demais); o resto são nulos disfarçados (Não Informado, SEM INFORMAÇÃO, 0, -, Null, Não se aplica) e 196 vazios |
| quantidade_homologada | Quantidade efetivamente comprada | Usada no cálculo dos valores totais (regra 5) |
| valor_unitario_homologado | Preço unitário pago | Formatos misturados, com `R$` e espaços invisíveis |
| valor_total_homologado | Valor total pago | Formatos misturados (`0,0001`); será recalculado (regra 5) |
| data_resultado | Data da homologação | Vários formatos; datas em 2026; pode ser anterior à publicação (regra 7) |
| data_cancelamento | Data de cancelamento do resultado | Só 62 preenchidos; preenchido = cancelado, descartar antes de escolher o vencedor (regra 2) |


---
## Perguntas da seção 7.4
### 1. Há `numero_controle_pncp` repetido em `compras`?


In [23]:
print('numero_controle_pncp repetidos = ' + str(compras["numero_controle_pncp"].duplicated().sum()))


numero_controle_pncp repetidos = 0


**Resposta:** não há repetição. As 2.042 linhas têm 2.042 `numero_controle_pncp` diferentes: cada linha é um edital único, e a coluna pode ser usada como chave.


### 2. Que formatos de dinheiro aparecem?


In [27]:
formatos = itens["valor_unitario_estimado"].str.replace(r"\d+", "9", regex=True)
formatos.value_counts(dropna=False).rename(index=repr)


valor_unitario_estimado
'9.9'                  5475
'R$ 9,9'                386
'R$ 9.9,9'              201
'R$\xa09,9\xa0'         193
'9,9'                   190
' R$ 9,9 '              184
'R$\xa09.9,9\xa0'        95
'9,9.9'                  87
'9.9,9'                  87
' R$ 9.9,9 '             59
'R$ 9.9.9,9'             39
'9,9,9.9'                25
'9.9.9,9'                24
' R$ 9.9.9,9 '           16
'R$\xa09.9.9,9\xa0'      13
'9,9,9,9.9'               1
'R$ 9.9.9.9,9'            1
' R$ 9.9.9.9,9 '          1
Name: count, dtype: int64

**Resposta:** 18 formatos diferentes na coluna `valor_unitario_estimado`:
- com e sem `R$`;
- decimal com ponto (`3.6`) ou com vírgula (`3,54`);
- separador de milhar com ponto (`1.693,46`) ou com vírgula (`20,400.00`): o padrão brasileiro e o americano misturados;
- espaços no início e espaços invisíveis (`\xa0`) entre o `R$` e o número.

Para converter, a regra do enunciado: um símbolo sozinho é sempre o decimal; símbolo repetido é milhar; com os dois, o último é o decimal.


### 3. Que formatos de data aparecem?


In [28]:
formatos_data = compras["data_publicacao"].str.replace(r"\d", "9", regex=True)
formatos_data.value_counts(dropna=False)


data_publicacao
9999-99-99T99:99:99    1319
99-99-9999              138
99/99/9999 99:99        126
99.99.9999              120
99/99/9999              117
9999/99/99              116
99999999                106
Name: count, dtype: int64

In [29]:
formatos_data_resultado = resultados["data_resultado"].str.replace(r"\d", "9", regex=True)
formatos_data_resultado.value_counts(dropna=False)


data_resultado
9999-99-99          10301
9999/99/99            960
99.99.9999            937
99/99/9999 99:99      934
99/99/9999            929
99999999              923
99-99-9999            912
Name: count, dtype: int64

**Resposta:** 7 formatos em `data_publicacao` e 7 em `data_resultado`: ISO com e sem hora (`2025-01-06T11:51:06`, `2025-03-10`), com barras (`2025/03/10`, `10/03/2025`, `10/03/2025 14:20`), com traço (`10-03-2025`), com ponto (`10.03.2025`) e sem separador (`20250310`). Para converter, a hora é descartada e cada formato é testado até um funcionar. Nos formatos com dia e mês, considera-se o dia primeiro (padrão brasileiro).


### 4. Que formatos de CNPJ/CPF aparecem?


In [30]:
compras["cnpj_orgao"].str.len().value_counts()


cnpj_orgao
14    1658
18     176
16     131
12      64
13      10
11       3
Name: count, dtype: int64

In [32]:
formatos = compras["cnpj_orgao"].str.replace(r"\d+", "9", regex=True)
formatos.value_counts(dropna=False).rename(index=repr)

cnpj_orgao
'9'            1735
'9.9.9/9-9'     176
'\xa09 '        131
Name: count, dtype: int64

In [35]:
resultados["ni_fornecedor"].str.len().value_counts()

ni_fornecedor
14    9612
11    3688
18    1240
16     698
13     468
10     154
12      18
9       15
8        2
7        1
Name: count, dtype: int64

In [37]:
formatos = resultados["ni_fornecedor"].str.replace(r"\d+", "9", regex=True)
formatos.value_counts(dropna=False).rename(index=repr)

ni_fornecedor
'9'                 13259
'9.9.9/9-9'          1239
'\xa09 '              988
'9.9.9-9'             392
'EX9'                   6
'ESTRANG9'              4
'\xa0ESTRANG9 '         2
'PF9'                   2
'ES.TRA.NG9/9-9'        1
'US9'                   1
'DUNS-9'                1
'EX9AGIA'               1
Name: count, dtype: int64

In [38]:
formatos_ni = resultados["ni_fornecedor"].str.replace(r"\d", "9", regex=True).map(repr)
pd.crosstab(formatos_ni, resultados["tipo_pessoa"])


tipo_pessoa,PE,PF,PJ
ni_fornecedor,,,
'99.999.999/9999-99',0,0,1239
'999.999.999-99',0,392,0
'99999999',1,1,0
'999999999',2,4,0
'9999999999',0,154,0
'99999999999',0,3687,1
'999999999999',0,0,18
'9999999999999',0,0,175
'99999999999999',0,0,9216


**Resposta:**

**`cnpj_orgao`** (órgãos são sempre pessoa jurídica, 14 dígitos): 1.658 limpos e 384 com problema:
- 176 com máscara (`00.000.000/0000-00`);
- 131 com espaço invisível (`\xa0`) no início e espaço comum no fim;
- 77 com zeros perdidos à esquerda (12, 13 ou 11 dígitos). Os de 11 dígitos têm o tamanho de um CPF, mas são CNPJs que perderam 3 zeros.

**`ni_fornecedor`**: o formato correto depende do `tipo_pessoa`:
- **PJ (CNPJ, 14 dígitos):** os mesmos problemas do órgão: máscara (1.239), espaços (695) e zeros perdidos (194).
- **PF (CPF, 11 dígitos):** máscara `000.000.000-00` (392), espaços (293) e zeros perdidos (159).
- **PE (estrangeiro):** identificadores sem formato padrão (`EX2544919`, `US95543`, `DUNS-...`). Não há como validar nem completar zeros; são mantidos como vieram, só sem espaços e em maiúsculas.

**Tratamento:** remover espaços e máscara, manter só os dígitos e completar com zeros à esquerda até 14 (PJ) ou 11 (PF). O `tipo_pessoa` decide o tamanho, porque só o número de dígitos não basta. Sem isso, o mesmo fornecedor apareceria com documentos diferentes e seria contado mais de uma vez.


### 5. Quais colunas têm nulos disfarçados (N/A, -, 0, vazio)?


In [39]:
disfarces = ["N/A", "NULL", "Null", "-", "0", "SEM INFORMAÇÃO", "NÃO SE APLICA", "Não se aplica", "Não Informado", "NÃO INFORMADO"]

(compras.isin(disfarces)).sum()


numero_controle_pncp            0
ano_compra                      0
sequencial_compra               0
numero_compra                   0
processo                       49
objeto_compra                   0
srp                             0
cnpj_orgao                      0
razao_social_orgao              0
poder_id                        0
esfera_id                       0
uf_sigla                        0
uf_nome                         0
municipio                       0
cod_ibge                        0
nome_unidade                    0
modalidade_id                   0
modalidade_nome                 0
modo_disputa                  941
situacao_compra                 0
valor_total_estimado            0
valor_total_homologado          0
data_publicacao                 0
data_abertura_proposta          0
data_encerramento_proposta      0
dtype: int64

In [40]:
(itens.isin(disfarces)).sum()


numero_controle_pncp          0
numero_item                   0
descricao                     0
material_ou_servico           0
categoria_item             7077
quantidade                    0
unidade_medida                0
valor_unitario_estimado       0
valor_total_estimado          0
sigiloso                      0
situacao_item               128
tem_resultado                 0
catalogo_codigo_item          0
criterio_julgamento        1883
dtype: int64

In [41]:
(resultados.isin(disfarces)).sum()


numero_controle_pncp            0
numero_item                     0
sequencial_resultado            0
ordem_classificacao_srp         0
ni_fornecedor                   0
tipo_pessoa                     0
nome_fornecedor                 0
porte                        4697
quantidade_homologada           0
valor_unitario_homologado       0
valor_total_homologado          0
data_resultado                  0
data_cancelamento               0
dtype: int64

**Resposta:** nulos disfarçados aparecem em 6 colunas, e o `.info()` sozinho não os detecta, porque conta esses textos como preenchidos:

| Tabela | Coluna | Disfarçados | Vazios reais |
|---|---|---|---|
| compras | processo | 49 | 28 |
| compras | modo_disputa | 941 | 26 |
| itens | categoria_item | 7.077 (coluna inteira) | 0 |
| itens | situacao_item | 128 | 92 |
| itens | criterio_julgamento | 1.883 | 0 |
| resultados | porte | 4.697 | 196 |

O mesmo "vazio" aparece de várias formas (`-`, `0`, `Null`, `SEM INFORMAÇÃO`, `NÃO INFORMADO`, `Não se aplica`). Na limpeza, todos viram nulo de verdade. O `0` só é tratado como nulo em colunas de texto; em quantidades e valores, 0 é número. Nenhuma dessas colunas entra nas 6 perguntas, mas `porte` sem informação em 31% dos fornecedores limitaria uma análise por porte de empresa.


### 6. Existe acento quebrado (Ã§, Ã£)? Em quais colunas?


In [42]:
compras.apply(lambda coluna: coluna.str.contains("Ã", na=False)).sum()


numero_controle_pncp            0
ano_compra                      0
sequencial_compra               0
numero_compra                   0
processo                       15
objeto_compra                 167
srp                             0
cnpj_orgao                      0
razao_social_orgao              0
poder_id                        0
esfera_id                       0
uf_sigla                        0
uf_nome                         0
municipio                      28
cod_ibge                        0
nome_unidade                  173
modalidade_id                   0
modalidade_nome                 0
modo_disputa                   15
situacao_compra                 0
valor_total_estimado            0
valor_total_homologado          0
data_publicacao                 0
data_abertura_proposta          0
data_encerramento_proposta      0
dtype: int64

In [43]:
compras.loc[compras["processo"].str.contains("Ã", na=False), "processo"].head()


122    SEM INFORMAÇÃO
170     NÃO INFORMADO
183    SEM INFORMAÇÃO
257    SEM INFORMAÇÃO
350    SEM INFORMAÇÃO
Name: processo, dtype: str

In [45]:
padrao_quebrado = r"[ÃÂ][\x80-\xbf]"
compras.apply(lambda coluna: coluna.str.contains(padrao_quebrado, na=False)).sum()


numero_controle_pncp           0
ano_compra                     0
sequencial_compra              0
numero_compra                  0
processo                       0
objeto_compra                 43
srp                            0
cnpj_orgao                     0
razao_social_orgao             0
poder_id                       0
esfera_id                      0
uf_sigla                       0
uf_nome                        0
municipio                     28
cod_ibge                       0
nome_unidade                  14
modalidade_id                  0
modalidade_nome                0
modo_disputa                   0
situacao_compra                0
valor_total_estimado           0
valor_total_homologado         0
data_publicacao                0
data_abertura_proposta         0
data_encerramento_proposta     0
dtype: int64